# 📊 Multivariate Regression Analysis — India Macroeconomic Variables (2015–2025)

**Method:** OLS (Ordinary Least Squares)  
**Dependent Variable:** GDP Growth Rate (%)  
**Independent Variables:** CPI Inflation, Repo Rate, Investment, Unemployment, Govt Debt, Govt Spending, Fiscal Deficit  
**Period:** 2015–16 to 2024–25  
**Sources:** RBI DBIE | MOSPI | World Bank | IMF WEO | Ministry of Finance

---
> *This notebook replicates the full regression analysis. Results include OLS coefficients, model diagnostics, VIF checks, and all visualizations.*


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
import warnings
warnings.filterwarnings("ignore")

print("✅ Libraries loaded successfully")


## 1. Dataset

Annual macroeconomic data for India from FY 2015–16 to 2024–25, compiled from official sources.

In [ ]:
df = pd.read_csv("india_macro_data.csv")
print(f"Shape: {df.shape[0]} observations × {df.shape[1]} variables\n")
df.style.highlight_min(subset=["GDP_Growth"], color="#ffd6d6")\
        .highlight_max(subset=["GDP_Growth"], color="#d6f5d6")\
        .format(precision=2)


## 2. Descriptive Statistics

In [ ]:
df.describe().round(2)


## 3. The Model

**Equation:**

$$GDP\_Growth = \beta_0 + \beta_1(CPI) + \beta_2(Repo) + \beta_3(Investment) + \beta_4(Unemployment) + \beta_5(GovtDebt) + \beta_6(GovtSpending) + \beta_7(FiscalDeficit) + \varepsilon$$

OLS minimises $\sum(Y_i - \hat{Y}_i)^2$ to find the best-fitting $\beta$ coefficients.


In [ ]:
y = df["GDP_Growth"]
X_cols = ["CPI_Inflation", "Repo_Rate", "Investment_GDP",
          "Unemployment", "Govt_Debt_GDP", "Govt_Spending_GDP", "Fiscal_Deficit_GDP"]
X = sm.add_constant(df[X_cols])

model = sm.OLS(y, X).fit()
print(model.summary())


## 4. Multicollinearity Check — Variance Inflation Factor (VIF)

VIF > 10 indicates potential multicollinearity. With only 10 observations and 7 predictors, high VIF is expected — a known limitation of this dataset size.

In [ ]:
vif_df = pd.DataFrame({
    "Variable": X_cols,
    "VIF": [variance_inflation_factor(X.values, i+1) for i in range(len(X_cols))]
}).sort_values("VIF", ascending=False).reset_index(drop=True)

vif_df["Note"] = vif_df["VIF"].apply(
    lambda v: "🔴 High" if v > 10 else ("🟡 Moderate" if v > 5 else "🟢 OK"))
print(vif_df.to_string(index=False))


## 5. Results & Visualizations

### Chart 1 — Actual vs Fitted GDP Growth

In [ ]:
from IPython.display import Image
Image("chart1_actual_vs_fitted.png", width=750)


### Chart 2 — OLS Regression Coefficients

In [ ]:
Image("chart2_coefficients.png", width=700)


### Chart 3 — Correlation Matrix

In [ ]:
Image("chart3_correlation_heatmap.png", width=620)


### Chart 4 — Macroeconomic Dashboard

In [ ]:
Image("chart4_macro_dashboard.png", width=800)


### Chart 5 — OLS Diagnostic Plots

In [ ]:
Image("chart5_diagnostics.png", width=750)


## 6. Key Findings & Interpretation

| Variable | β Coefficient | Theory Validated |
|----------|--------------|-----------------|
| CPI Inflation | –0.74 | AD-AS: inflation reduces output |
| Repo Rate | –0.91 | IS-LM: monetary tightening slows growth |
| Investment | +0.48 | Keynesian Multiplier: investment drives growth |
| Unemployment | –1.38 | **Okun's Law** — strongest coefficient |
| Govt Debt | –0.11 | Crowding-out of private credit |
| Govt Spending | +0.32 | Short-run fiscal multiplier |
| Fiscal Deficit | –0.52 | Crowding-out effect |

**Model Fit:**
- R² = 0.928 → explains 92.8% of GDP variation
- Adjusted R² = 0.678 → corrected for 7 predictors
- COVID-19 (2020–21) is a clear structural break — captured by the model

**Important Note on Small Sample:**  
With n=10 and k=7 predictors, the model has only 2 degrees of freedom for residuals. High VIF values indicate multicollinearity among fiscal variables (Debt, Spending, Deficit are highly correlated, r > 0.94). This is a known limitation of annual macro data — a longer time series would improve reliability.

---
> *"All models are wrong, but some are useful."* — George E. P. Box
